In [ ]:
# run_raster_parallel.py
from __future__ import annotations

import os
import multiprocessing as mp
import concurrent.futures as cf
from pathlib import Path

from general_utils import find_ephys_sessions
from raster_worker import process_session, OUTDIR  # import the top-level worker

def main():
    OUTDIR.mkdir(parents=True, exist_ok=True)
    _, _, sessions = find_ephys_sessions()
    print("Sessions:", sessions)
    if not sessions:
        return

    # Use spawn to avoid HDF5/NWB fork-safety issues
    ctx = mp.get_context("spawn")
    max_workers = min(len(sessions), os.cpu_count() or 1)
    print(f"Running in parallel with {max_workers} workers (spawn)")

    with cf.ProcessPoolExecutor(max_workers=max_workers, mp_context=ctx) as ex:
        futures = {ex.submit(process_session, s): s for s in sessions}
        for fut in cf.as_completed(futures):
            print(fut.result())

if __name__ == "__main__":
    # If something else already set the start method, this is fine.
    try:
        mp.set_start_method("spawn", force=False)
    except RuntimeError:
        pass
    main()


In [ ]:
# move files to different subfolders

from pathlib import Path
import re
import shutil

ROOT = Path("/root/capsule/scratch/raster_plot")
DRY_RUN = False  # change to False after checking

PNG_RE = re.compile(r"^(?P<prefix>.+)_unit_(?P<unit>\d+)\.png$", re.IGNORECASE)

moved = 0
skipped = 0

for session_dir in sorted([p for p in ROOT.iterdir() if p.is_dir()]):

    for png_path in sorted(session_dir.glob("*.png")):
        m = PNG_RE.match(png_path.name)
        if not m:
            skipped += 1
            continue

        prefix = m.group("prefix")

        # ✅ destination is directly under session
        dest_dir = session_dir / prefix
        dest_dir.mkdir(parents=True, exist_ok=True)

        dest_path = dest_dir / png_path.name

        if DRY_RUN:
            print(f"[DRY] {png_path} -> {dest_path}")
        else:
            shutil.move(str(png_path), str(dest_path))
            print(f"[MOVED] {png_path} -> {dest_path}")

        moved += 1

print("\nSummary")
print("Moved:", moved)
print("Skipped:", skipped)


In [1]:
from pathlib import Path
import re
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

ROOT = Path("/root/capsule/scratch/raster_plot")
DRY_RUN = False

PNG_RE = re.compile(r"^(?P<prefix>.+)_unit_(?P<unit>\d+)\.png$", re.IGNORECASE)

def move_one(png_path: Path, dest_path: Path) -> None:
    if DRY_RUN:
        return
    png_path.rename(dest_path)

moved = 0
skipped = 0

max_workers = min(32, (os.cpu_count() or 8) * 2)
PRINT_EVERY = 500  # adjust

for session_dir in (p for p in ROOT.iterdir() if p.is_dir()):
    created = set()
    tasks = []

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for png_path in session_dir.glob("*.png"):
            m = PNG_RE.match(png_path.name)
            if not m:
                skipped += 1
                continue

            prefix = m.group("prefix")
            dest_dir = session_dir / prefix

            if prefix not in created:
                if not DRY_RUN:
                    dest_dir.mkdir(parents=True, exist_ok=True)
                created.add(prefix)

            dest_path = dest_dir / png_path.name
            tasks.append(ex.submit(move_one, png_path, dest_path))

        total = len(tasks)
        done = 0

        for fut in as_completed(tasks):
            fut.result()
            moved += 1
            done += 1

            if done % PRINT_EVERY == 0 or done == total:
                print(f"[{session_dir.name}] {done}/{total} done | moved_total={moved} skipped={skipped}")

print("\nSummary")
print("Moved:", moved)
print("Skipped:", skipped)


[ecephys_764791_2025-01-16_12-50-17_sorted_2025-04-25_14-11-37] 500/61293 done | moved_total=500 skipped=0
[ecephys_764791_2025-01-16_12-50-17_sorted_2025-04-25_14-11-37] 1000/61293 done | moved_total=1000 skipped=0
[ecephys_764791_2025-01-16_12-50-17_sorted_2025-04-25_14-11-37] 1500/61293 done | moved_total=1500 skipped=0
[ecephys_764791_2025-01-16_12-50-17_sorted_2025-04-25_14-11-37] 2000/61293 done | moved_total=2000 skipped=0
[ecephys_764791_2025-01-16_12-50-17_sorted_2025-04-25_14-11-37] 2500/61293 done | moved_total=2500 skipped=0
[ecephys_764791_2025-01-16_12-50-17_sorted_2025-04-25_14-11-37] 3000/61293 done | moved_total=3000 skipped=0
[ecephys_764791_2025-01-16_12-50-17_sorted_2025-04-25_14-11-37] 3500/61293 done | moved_total=3500 skipped=0
[ecephys_764791_2025-01-16_12-50-17_sorted_2025-04-25_14-11-37] 4000/61293 done | moved_total=4000 skipped=0
[ecephys_764791_2025-01-16_12-50-17_sorted_2025-04-25_14-11-37] 4500/61293 done | moved_total=4500 skipped=0
[ecephys_764791_2025-